# Загрузка данных

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
from scipy.sparse import csr_matrix
from datetime import timedelta

In [2]:
interactions = pd.read_csv('/Users/admin/Desktop/RecSys/KION_DATASET/interactions.csv')
items = pd.read_csv('/Users/admin/Desktop/RecSys/KION_DATASET/data_original/items.csv')

In [3]:
interactions['last_watch_dt'].unique()
interactions['last_watch_dt'] = pd.to_datetime(interactions['last_watch_dt'], format='%Y-%m-%d', errors='coerce')
interactions = interactions.dropna(subset=['last_watch_dt'])

In [4]:
size_test = 1
data = interactions[(interactions['watched_pct']>50.0)&(interactions['total_dur']>300.0)]
items_data = data['item_id'].value_counts()
active_items = items_data[items_data > 10].index
data = data[data['item_id'].isin(active_items)]
data_train = data[data['last_watch_dt'] < data['last_watch_dt'].max() - timedelta(weeks=size_test)]
data_test = data[data['last_watch_dt'] >= data['last_watch_dt'].max() - timedelta(weeks=size_test)]

In [5]:
monthly_train = data_train.groupby(pd.Grouper(key='last_watch_dt', freq='M')).agg({
    'user_id': 'count',
    'total_dur': 'sum'
}).rename(columns={'user_id': 'total_interactions'})
monthly_train

/var/folders/qs/gxy4f99x26l7w88l0zk12_w00000gn/T/ipykernel_1395/491339101.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_train = data_train.groupby(pd.Grouper(key='last_watch_dt', freq='M')).agg({


,total_interactions,total_dur
last_watch_dt,,
2021-03-31,32908,3.469953e+08
2021-04-30,62824,6.823194e+08
2021-05-31,93267,1.230526e+09
2021-06-30,162848,2.826922e+09
2021-07-31,184508,3.054232e+09
2021-08-31,78760,1.169614e+09


In [6]:
train_sorted = data_train.sort_values(['user_id', 'last_watch_dt'])
    
train = train_sorted.groupby('user_id').tail(10)

In [7]:
test_users = set(data_test['user_id'].unique())
train_users = set(train['user_id'].unique())
common_users = test_users & train_users
    
print(f"Пользователей в test: {len(test_users)}")
print(f"Пользователей в train: {len(train_users)}")
print(f"Общих пользователей: {len(common_users)}")

Пользователей в test: 36175
Пользователей в train: 284662
Общих пользователей: 17114


In [8]:
user_interaction_test = data_test['user_id'].value_counts()
active_users_test = user_interaction_test[user_interaction_test > 2].index
test_active = data_test[data_test['user_id'].isin(active_users_test)]

user_interaction_train = train['user_id'].value_counts()
active_users_train = user_interaction_train[user_interaction_train > 2].index
train_active = train[train['user_id'].isin(active_users_train)]

test_users = test_active['user_id'].unique()
train_users = train_active['user_id'].unique()

data_train = train_active[train_active['user_id'].isin(test_users)]
data_test = test_active[test_active['user_id'].isin(train_users)]

In [9]:
test_users = set(data_test['user_id'].unique())
train_users = set(data_train['user_id'].unique())
common_users = test_users & train_users
    
print(f"Пользователей в test: {len(test_users)}")
print(f"Пользователей в train: {len(train_users)}")
print(f"Общих пользователей: {len(common_users)}")

Пользователей в test: 1120
Пользователей в train: 1120
Общих пользователей: 1120


# Random

In [10]:
data_test_group = data_test.groupby(data_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

,user_id,item_id
0,2756,"[1247, 2925, 13325, 7793, 15399, 14488]"
1,4027,"[5732, 3838, 13159, 3349]"
2,4249,"[13483, 9778, 10125, 2865]"
3,4997,"[3190, 5732, 9194]"
4,5119,"[12356, 13849, 2596, 7968]"
...,...,...
1115,1093424,"[10636, 14488, 11118]"
1116,1094628,"[13543, 125, 10119, 9194, 3518]"
1117,1095418,"[14317, 2025, 1290]"
1118,1097444,"[13650, 12841, 12250, 2483]"


In [11]:
users_unique_test = data_test_group['user_id']
items_unique = interactions['item_id'].unique()

size_recommend = 10

recommendations = []
for user in users_unique_test:
    random_items = np.random.choice(items_unique, size=size_recommend, replace=False)
    recommendations.append(random_items)

result_random = data_test_group.copy()
result_random['random_recommendation'] = recommendations

In [12]:
result_random

,user_id,item_id,random_recommendation
0,2756,"[1247, 2925, 13325, 7793, 15399, 14488]","[11407, 14295, 3411, 10900, 11648, 9767, 9681,..."
1,4027,"[5732, 3838, 13159, 3349]","[10110, 11014, 325, 6483, 11285, 8741, 11610, ..."
2,4249,"[13483, 9778, 10125, 2865]","[13033, 12591, 12963, 13418, 5651, 7396, 8532,..."
3,4997,"[3190, 5732, 9194]","[6423, 4916, 715, 6745, 11294, 15971, 8519, 83..."
4,5119,"[12356, 13849, 2596, 7968]","[15662, 9193, 14775, 14142, 3130, 2898, 4113, ..."
...,...,...,...
1115,1093424,"[10636, 14488, 11118]","[8618, 10995, 7684, 12336, 14000, 1468, 3192, ..."
1116,1094628,"[13543, 125, 10119, 9194, 3518]","[3554, 5081, 9810, 812, 11314, 4299, 6317, 600..."
1117,1095418,"[14317, 2025, 1290]","[12185, 6500, 1469, 3204, 4807, 1303, 8134, 11..."
1118,1097444,"[13650, 12841, 12250, 2483]","[8351, 8690, 7142, 14597, 1079, 4837, 10530, 8..."


In [13]:
def precision(recommended_list, bought_list):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    return precision


def precision_at_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    bought_list = bought_list 
    recommended_list = recommended_list[:k]
    
    flags = np.isin(bought_list, recommended_list)
    
    precision = flags.sum() / len(recommended_list)
    
    
    return precision

def ap_k(recommended_list, bought_list, k=5):
    
    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)
    
    flags = np.isin(recommended_list, bought_list)
    
    if sum(flags) == 0:
        return 0
    
    sum_ = 0
    for i in range(0, k-1):
        if flags[i] == True:
            p_k = precision_at_k(recommended_list, bought_list, k=i+1)
            sum_ += p_k
            
    result = sum_ / sum(flags)
    
    return result

def map_k(recommended_list, bought_list, k=5, u=1):
    
    # your_code
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

In [14]:
print('random map_k =', map_k(result_random['random_recommendation'], result_random['item_id'], k=10, u=len(result_random)))

random map_k = 0.000744047619047619


# Popular

In [15]:
def popularity(train, n=10):
    train_group = train.groupby(train['item_id'])['user_id'].unique().reset_index()
    train_group.columns = ['item_id', 'user_id']

    train_group['user_sum'] = train_group['user_id'].apply(len)
    top_10_train = train_group.nlargest(n, 'user_sum')
    top_10 = top_10_train['item_id']

    return top_10.tolist()

popular = popularity(data_train, n=10)
result_popular = data_test_group.copy()
result_popular['popular_recommendation'] = result_popular['user_id'].apply(lambda x: popular)

In [16]:
result_popular

,user_id,item_id,popular_recommendation
0,2756,"[1247, 2925, 13325, 7793, 15399, 14488]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
1,4027,"[5732, 3838, 13159, 3349]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
2,4249,"[13483, 9778, 10125, 2865]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
3,4997,"[3190, 5732, 9194]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
4,5119,"[12356, 13849, 2596, 7968]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
...,...,...,...
1115,1093424,"[10636, 14488, 11118]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
1116,1094628,"[13543, 125, 10119, 9194, 3518]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
1117,1095418,"[14317, 2025, 1290]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."
1118,1097444,"[13650, 12841, 12250, 2483]","[9728, 13865, 3734, 15297, 10440, 3182, 7571, ..."


In [17]:
print('popular map_k =', map_k(result_popular['popular_recommendation'], result_popular['item_id'], k=10, u=len(result_popular)))

popular map_k = 0.05546875000000001


In [18]:
data_test.to_csv('test_interactions.csv', index=False)
data_train.to_csv('train_interactions.csv', index=False)

In [19]:
random_user = np.random.choice(data_test['user_id'], replace=False)

In [20]:
rec_random = result_random.loc[result_random['user_id']==random_user, 'random_recommendation'].iloc[0]
rec_popular = result_popular.loc[result_popular['user_id']==random_user, 'popular_recommendation'].iloc[0]
rec_test = data_test_group.loc[data_test_group['user_id']==random_user, 'item_id'].iloc[0]

random_user_item = []
popular_user_item = []
test_user_item = []

for i in range(len(rec_random)):
    title_item = items.loc[items['item_id']==rec_random[i], 'title'].iloc[0]
    random_user_item.append(title_item)

for i in range(len(rec_popular)):
    title_item = items.loc[items['item_id']==rec_popular[i], 'title'].iloc[0]
    popular_user_item.append(title_item)

for i in range(len(rec_test)):
    title_item = items.loc[items['item_id']==rec_test[i], 'title'].iloc[0]
    test_user_item.append(title_item)

print(f"Рекомендаций для пользователя {random_user}: \n Random") 
for i, (item) in enumerate(zip(random_user_item)):
    print(f"{i+1}. {item}")
print('Popular')
for i, (item) in enumerate(zip(popular_user_item)):
    print(f"{i+1}. {item}")
print('Test')
for i, (item) in enumerate(zip(test_user_item)):
    print(f"{i+1}. {item}")


Рекомендаций для пользователя 935752: 
 Random
1. ('Легенды Нетайи',)
2. ('Радио Романтика',)
3. ('Айрис',)
4. ('Ночная фиалка',)
5. ('Королева бандитов 2',)
6. ('Помни меня',)
7. ('Новички 90+',)
8. ('Вторжение',)
9. ('Домохозяйки',)
10. ('Нехожеными тропами Ингушетии',)
Popular
1. ('Гнев человеческий',)
2. ('Девятаев',)
3. ('Прабабушка легкого поведения',)
4. ('Клиника счастья',)
5. ('Хрустальный',)
6. ('Ральф против Интернета',)
7. ('100% волк',)
8. ('Моана',)
9. ('Зверополис',)
10. ('Тайна Коко',)
Test
1. ('Букашки 2',)
2. ('Собибор',)
3. ('Ледниковый период 4: Континентальный дрейф',)
